# Phase 0 - PDH/PDL Touch Event Study (measurement only, no trades)

**Purpose:** reproduce or refute, on LEAN data:
1. The claimed invasion magnitudes: E[delta-HL] ~ 0.12-0.18 x YR after touching yesterday's high (mirror for the low).
2. Window-vs-change: does invasion depth/stability vary by touch time of day (does the 11:00-14:00 stabilization exist as an output)?
3. VIX-vs-change: correlation of invasion depth with VIX level and changes in VIX.

**Method:** every regular-hours SPY session 2021-2026, detect the first touch of PDH and of PDL at ANY time (window-free), log the full feature matrix, then report descriptive correlations. No orders, no strategy logic.

In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd

qb = QuantBook()
spy = qb.add_equity("SPY", Resolution.MINUTE, data_normalization_mode=DataNormalizationMode.RAW).symbol
vix = qb.add_index("VIX").symbol

START = datetime(2021, 1, 1)
END   = datetime(2026, 10, 1)
NY    = "America/New_York"

def to_ny(idx):
    idx = pd.to_datetime(idx)
    return idx.tz_convert(NY) if idx.tz is not None else idx.tz_localize(NY)

print("Setup OK:", spy, vix, START.date(), "->", END.date())

In [ ]:
# --- Daily context: prior-day levels, range, ATR proxy, VIX daily ---
daily = qb.history(spy, START, END, Resolution.DAILY)
dbars = daily.loc[spy].copy()
dbars.index = to_ny(dbars.index)
dbars["date"] = [d.date() for d in dbars.index]
dbars = dbars.set_index("date").sort_index()
dbars["YR"] = dbars["high"] - dbars["low"]
dbars["range14"] = dbars["YR"].rolling(14).mean()      # crude ATR proxy on true ranges
dbars["vix_prev"] = None

vix_d = qb.history(vix, START, END, Resolution.DAILY)
vd = vix_d.loc[vix].copy()
vd.index = to_ny(vd.index)
vd["date"] = [d.date() for d in vd.index]
vd = vd.set_index("date").sort_index()
vd["vix_close"] = vd["close"]
vd["dvix_prevclose"] = vd["vix_close"].diff()          # change vs T-1 close

dbars = dbars.join(vd[["vix_close", "dvix_prevclose"]], how="left")
print("daily sessions:", len(dbars), "| VIX sessions:", vd["vix_close"].notna().sum())
dbars[["high", "low", "close", "YR", "range14", "vix_close"]].tail(3)

In [ ]:
# --- Event engine v2: fixed rej30 (30-bar window), time_above (close-based), acc20 ---
dbars["vix_prev_day"] = dbars["vix_close"].shift(1)
BYEAR = sorted({d.year for d in dbars.index})

def extract_day(d, day):
    if len(day) < 300:
        return []
    row = dbars.loc[d]
    PDH, PDL, YR = row["high"], row["low"], row["YR"]
    if not np.isfinite(YR) or YR <= 0:
        return []
    h = day["high"].values; l = day["low"].values
    c = day["close"].values; o = day["open"].values; v = day["volume"].values
    t = day.index
    n = len(t)
    hi_run = np.maximum.accumulate(h)
    lo_run = np.minimum.accumulate(l)
    day_open = o[0]
    cut14 = t[0].replace(hour=14, minute=0)
    i14 = int(np.searchsorted(t, cut14))
    vix_prev = row["vix_prev_day"]

    out = []
    for side in ("PDH", "PDL"):
        i0 = int(np.argmax(hi_run >= PDH)) if side == "PDH" and (hi_run >= PDH).any() else -1
        if side == "PDL":
            i0 = int(np.argmax(lo_run <= PDL)) if (lo_run <= PDL).any() else -1
        if i0 < 0:
            continue
        lvl = PDH if side == "PDH" else PDL
        path = l if side == "PDH" else h
        def inv_depth(j_end):
            seg = path[i0:j_end]
            if len(seg) == 0: return np.nan
            worst = seg.min() if side == "PDH" else seg.max()
            return (lvl - worst) / YR if side == "PDH" else (worst - lvl) / YR
        d_close = inv_depth(n)
        d_1400  = inv_depth(max(i14, i0 + 1))
        d_30 = inv_depth(min(n, i0 + 30)); d_60 = inv_depth(min(n, i0 + 60)); d_120 = inv_depth(min(n, i0 + 120))
        seg_l = path[i0:]
        rel = int(np.argmax(seg_l.min() == seg_l)) if side == "PDH" else int(np.argmax(seg_l.max() == seg_l))
        tte = (t[i0 + rel] - t[i0]).total_seconds() / 60.0
        # FIXED: persistence measured on CLOSES vs the level
        crossed_back = np.where(c[i0:] < lvl)[0] if side == "PDH" else np.where(c[i0:] > lvl)[0]
        time_above = (t[i0 + crossed_back[0]] - t[i0]).total_seconds() / 60.0 if len(crossed_back) else (t[-1] - t[i0]).total_seconds() / 60.0
        seg20c = c[i0:min(n, i0 + 20)]
        acc20 = bool(seg20c.min() > lvl) if side == "PDH" else bool(seg20c.max() < lvl)
        hh = path[i0:]
        recross = int(np.sum((hh[1:] <= lvl) & (hh[:-1] > lvl))) if side == "PDH" else int(np.sum((hh[1:] >= lvl) & (hh[:-1] < lvl)))
        mae = (hh.max() - lvl) / YR if side == "PDH" else (lvl - hh.min()) / YR
        pre30 = v[max(0, i0 - 30):i0]
        rvol_touch = (v[i0] / pre30.mean()) if pre30.mean() > 0 else np.nan
        pre_sum = pre30.sum()
        cumrvol30 = v[i0:i0 + 30].sum() / pre_sum if pre_sum > 0 else np.nan
        pre5 = v[max(0, i0 - 5):i0].sum()
        prepost5 = v[i0:i0 + 5].sum() / pre5 if pre5 > 0 else np.nan
        seg_c = c[i0:i0 + rel + 1]; seg_o = o[i0:i0 + rel + 1]; seg_v = v[i0:i0 + rel + 1]
        imb = float(np.sum(seg_v * np.where(seg_c >= seg_o, 1.0, -1.0)) / seg_v.sum()) if seg_v.sum() > 0 else np.nan
        # FIXED: rejection measured within 30 BARS of the touch
        seg30 = path[i0:min(n, i0 + 30)]
        rej30 = bool(seg30.min() <= lvl - 0.02 * YR) if side == "PDH" else bool(seg30.max() >= lvl + 0.02 * YR)
        p_touch = c[i0]
        ret_1400 = (c[min(i14, n - 1)] / p_touch - 1) * 1e4 if i14 < n else np.nan
        ret_close = (c[-1] / p_touch - 1) * 1e4
        run = bool(c[-1] > PDH) if side == "PDH" else bool(c[-1] < PDL)
        dclass = "run" if run else ("rejection" if rej30 else "other")
        vt = vix_min.asof(t[i0]) if len(vix_min) else np.nan
        vt_ext = vix_min.asof(t[i0 + rel]) if len(vix_min) else np.nan
        vt_1400 = vix_min.asof(t[min(i14, n - 1)]) if (len(vix_min) and i14 < n) else np.nan
        vt_close = vix_min.asof(t[-1]) if len(vix_min) else np.nan
        out.append(dict(
            date=d, side=side, touch_time=t[i0], touch_min=(t[i0] - t[0]).total_seconds() / 60.0,
            gap_over_open=bool(day_open >= PDH) if side == "PDH" else bool(day_open <= PDL),
            YR=YR, range14=row["range14"], yr_atr=(YR / row["range14"]) if np.isfinite(row["range14"]) and row["range14"] > 0 else np.nan,
            dow=d.weekday(),
            dHL_close=d_close, dHL_1400=d_1400, dHL_30=d_30, dHL_60=d_60, dHL_120=d_120,
            tte_min=tte, time_above_min=time_above, acc20=acc20, recross=recross, mae_yr=mae,
            rvol_touch=rvol_touch, cumrvol30=cumrvol30, prepost5=prepost5, imb=imb, rej30=rej30,
            ret_1400_bps=ret_1400, ret_close_bps=ret_close, dclass=dclass,
            vix_touch=vt, vix_prev=vix_prev,
            dvix_to_touch=(vt - vix_prev) if np.isfinite(vt) and np.isfinite(vix_prev) else np.nan,
            dvix_to_ext=(vt_ext - vt) if np.isfinite(vt) and np.isfinite(vt_ext) else np.nan,
            dvix_to_1400=(vt_1400 - vt) if np.isfinite(vt) and np.isfinite(vt_1400) else np.nan,
            dvix_to_close=(vt_close - vt) if np.isfinite(vt) and np.isfinite(vt_close) else np.nan,
        ))
    return out

print("event engine v2 defined | years:", BYEAR)

In [ ]:
# --- Run the event study year by year (chunked to keep memory sane) ---
import time as _time
events, vix_chunks = [], []
vix_min = pd.Series(dtype=float)
t0 = _time.time()
for y in sorted(BYEAR):
    s, e = datetime(y, 1, 1), datetime(y, 12, 31)
    vm = qb.history(vix, s, e, Resolution.MINUTE)
    if len(vm):
        vchunk = vm.loc[vix]["close"]
        vchunk.index = to_ny(vchunk.index)
        vix_chunks.append(vchunk)
    vix_min = pd.concat(vix_chunks).sort_index()
    vix_min = vix_min[~vix_min.index.duplicated(keep="last")]
    mh = qb.history(spy, s, e, Resolution.MINUTE)
    if not len(mh):
        print(y, "no minute data"); continue
    m = mh.loc[spy].copy()
    m.index = to_ny(m.index)
    m = m.sort_index()
    m = m.between_time("09:31", "16:00")
    m["d"] = [ts.date() for ts in m.index]
    cnt = 0
    for d, day in m.groupby("d"):
        if d not in dbars.index:
            continue
        evs = extract_day(d, day)
        events.extend(evs); cnt += len(evs)
    print(f"{y}: {cnt} touch events  ({_time.time()-t0:.0f}s elapsed)")

ev = pd.DataFrame(events)
print("\nTOTAL events:", len(ev), "| PDH:", (ev.side == 'PDH').sum(), "| PDL:", (ev.side == 'PDL').sum())
print("VIX minute coverage:", len(vix_min), "bars")

In [ ]:
# --- Deliverable 1: headline invasion statistics vs the claimed 0.12-0.18 x YR ---
base = ev[~ev.gap_over_open].copy()          # base events: level not already breached at the open
gaps = ev[ev.gap_over_open].copy()
pdh, pdl = base[base.side == "PDH"], base[base.side == "PDL"]

def stats(s):
    return pd.Series({"n": len(s), "mean": s.mean(), "median": s.median(),
                      "q25": s.quantile(.25), "q75": s.quantile(.75),
                      "p10": s.quantile(.10), "p90": s.quantile(.90)})

print("=== Headline invasion depth (fraction of YR), gap-open days excluded ===")
summ = pd.DataFrame({"dHL after PDH touch": stats(pdh.dHL_close),
                     "dLH after PDL touch": stats(pdl.dHL_close)})
print(summ.round(3))
print("\nClaim check: E[dHL] in 0.12-0.18? ->", round(pdh.dHL_close.mean(), 3),
      "| E[dLH] in 0.05-0.09? ->", round(pdl.dHL_close.mean(), 3))
print("Gap-open events (tagged, excluded above):", len(gaps))

print("\n=== By year (PDH events) ===")
yr = pdh.groupby(pdh.date.map(lambda d: d.year)).dHL_close.apply(stats).unstack()
print(yr[["n", "mean", "median", "q25", "q75"]].round(3))

In [ ]:
# --- Deliverable 2: horizon profile + WINDOW-vs-CHANGE table ---
print("=== Invasion depth by horizon (PDH events, YR units) ===")
hor = pd.DataFrame({k: stats(pdh[k]) for k in ["dHL_30", "dHL_60", "dHL_120", "dHL_1400", "dHL_close"]}).T
print(hor.round(3))

def bucket(m):
    if m < 30:   return "09:31-10:00"
    if m < 90:   return "10:00-11:00"
    if m < 270:  return "11:00-14:00"
    if m < 330:  return "14:00-15:00"
    return "15:00-16:00"

base["wbucket"] = base.touch_min.map(bucket)
order = ["09:31-10:00", "10:00-11:00", "11:00-14:00", "14:00-15:00", "15:00-16:00"]
for name, sub in (("PDH", pdh), ("PDL", pdl)):
    g = sub.groupby(sub.touch_min.map(bucket)).agg(
        n=("dHL_close", "size"), med_dHL=("dHL_close", "median"), mean_dHL=("dHL_close", "mean"),
        q25=("dHL_close", lambda s: s.quantile(.25)), q75=("dHL_close", lambda s: s.quantile(.75)),
        med_tte=("tte_min", "median"), med_ret_bps=("ret_close_bps", "median"))
    g = g.reindex(order)
    g["iqr"] = g.q75 - g.q25
    print(f"\n=== WINDOW vs CHANGE ({name} events) ===")
    print(g[["n", "med_dHL", "mean_dHL", "iqr", "med_tte", "med_ret_bps"]].round(3))

In [ ]:
# --- Deliverable 3: VIX-vs-change + volume conditioning + monotonicity ---
print("=== Spearman correlations with dHL (PDH events) ===")
cols = ["touch_min", "rvol_touch", "cumrvol30", "prepost5", "imb",
        "vix_touch", "dvix_to_touch", "dvix_to_1400", "dvix_to_close", "dvix_to_ext"]
corr = pdh[["dHL_close"] + cols].corr(method="spearman")["dHL_close"].drop("dHL_close").sort_values()
print(corr.round(3))
print("\nNOTE: dvix_to_ext is near-tautological (VIX mirrors the SPX path to the extreme).")
print("Incremental tests are: vix_touch (level) and dvix_to_touch (pre-touch change).")

vb = pd.cut(pdh.vix_touch, [0, 15, 20, 25, 100], labels=["<15", "15-20", "20-25", ">25"])
print("\n=== VIX level bucket vs dHL (PDH) ===")
print(pdh.groupby(vb, observed=True).agg(n=("dHL_close", "size"), med=("dHL_close", "median"),
      mean=("dHL_close", "mean"), rej_rate=("rej30", "mean"), run_rate=("dclass", lambda s: (s == "run").mean())).round(3))

print("\n=== Volume conditioning: cumRVOL(30m) terciles vs dHL (PDH) ===")
tv = pd.qcut(pdh.cumrvol30, 3, labels=["low", "mid", "high"])
print(pdh.groupby(tv, observed=True).agg(n=("dHL_close", "size"), med=("dHL_close", "median"),
      rej_rate=("rej30", "mean")).round(3))

print("\n=== Monotonicity: early invasion predicts final invasion? ===")
from scipy.stats import spearmanr
for h in ["dHL_30", "dHL_60", "dHL_120"]:
    r, p = spearmanr(pdh[h], pdh.dHL_close)
    print(f"spearman({h}, dHL_close) = {r:.3f}  (p={p:.1e})")

print("\n=== Outcome classes (PDH) ===")
print(pdh.dclass.value_counts(normalize=True).round(3))
print("acceptance (held >=20 min):", round(pdh.acc20.mean(), 3), "| median time-above-level:", round(pdh.time_above_min.median(), 1), "min")

In [ ]:
# --- Diagnostics: alignment check, run recount, retrace threshold sweep ---
# 1) daily-vs-minute alignment on 3 sample dates
for d in [pdh.date.iloc[100], pdh.date.iloc[700], pdh.date.iloc[-1]]:
    mh = qb.history(spy, datetime(d.year, d.month, d.day), datetime(d.year, d.month, d.day) + timedelta(days=1), Resolution.MINUTE)
    if len(mh):
        mm = mh.loc[spy]
        print(d, "daily H/L:", round(dbars.loc[d, "high"], 2), round(dbars.loc[d, "low"], 2),
              "| minute H/L:", round(mm["high"].max(), 2), round(mm["low"].min(), 2))

# 2) threshold sweep from stored dHL_30 / dHL_close (PDH events)
print("\nP(retrace > x*YR within 30 min of touch) and P(by close):" )
for x in [0.02, 0.05, 0.10, 0.20, 0.30, 0.50, 1.00]:
    p30 = (pdh.dHL_30 > x).mean(); pcl = (pdh.dHL_close > x).mean()
    print(f"  x={x:4.2f}   30min: {p30:6.3f}   by close: {pcl:6.3f}")

# 3) acceptance / persistence from close-based time_above
print("\nP(close stays above level >= 20 min):", round((pdh.time_above_min >= 20).mean(), 4))
print("P(close stays above level >= 60 min):", round((pdh.time_above_min >= 60).mean(), 4))
print("touch bar closes back below level (close-based):", round((pdh.time_above_min == 0).mean(), 3))

# 4) run recount: does SPY ever close above PDH after touching it mid-session?
above = (pdh.ret_close_bps > 0)  # close above touch price (touch price ~= PDH within a tick)
print("\nPDH events closing above the touch price:", int(above.sum()), "of", len(pdh),
      f"({above.mean():.4f})")

In [ ]:
# --- Summary + persistence note ---
print("Events:", len(ev), "| PDH:", (ev.side == 'PDH').sum(), "| PDL:", (ev.side == 'PDL').sum(), "| gap-open tagged:", ev.gap_over_open.sum())
print("Columns:", list(ev.columns))
try:
    ev.to_csv("/tmp/phase0_spy_pdh_pdl_events.csv", index=False)
    print("local copy written to /tmp/phase0_spy_pdh_pdl_events.csv")
except Exception as ex:
    print("local write failed:", ex)
print("\nNOTE: org Object Store write not permitted for this account - re-run cells 3-4 (approx 10s) to rebuild `ev` in any session.")

In [ ]:
# --- Cell 10: checklist audit --- re-touches, RVOL-by-time, gap reconciliation ---
rows = []
for y in sorted(BYEAR):
    s, e = datetime(y,1,1), datetime(y,12,31)
    mh = qb.history(spy, s, e, Resolution.MINUTE)
    m = mh.loc[spy].copy(); m.index = to_ny(m.index); m = m.sort_index()
    m = m.between_time("09:31","16:00")
    m["d"] = [ts.date() for ts in m.index]
    for d, day in m.groupby("d"):
        if d not in dbars.index: continue
        row = dbars.loc[d]
        PDH, PDL, YR = row["high"], row["low"], row["YR"]
        if not (np.isfinite(YR) and YR > 0): continue
        h = day["high"].values; l = day["low"].values; c = day["close"].values
        t = day.index; n = len(t)
        for side, lvl in (("PDH", PDH), ("PDL", PDL)):
            above = side == "PDH"
            mask = (h >= lvl) if above else (l <= lvl)   # per-bar touch, not running max
            if not mask.any(): continue
            idx = np.flatnonzero(mask)
            eps = np.split(idx, np.where(np.diff(idx) > 1)[0] + 1)  # contiguous touch episodes
            path = l if above else h
            for k, ep in enumerate(eps, start=1):
                i0 = int(ep[0])
                def depth(j):
                    seg = path[i0:min(n, j)]
                    return ((lvl - seg.min())/YR if above else (seg.max() - lvl)/YR) if len(seg) else np.nan
                rows.append(dict(date=d, side=side, ep=k, n_ep=len(eps),
                    touch_min=(t[i0]-t[0]).total_seconds()/60.0,
                    d30=depth(i0+30), dcl=depth(n),
                    run=bool(c[-1] > PDH) if above else bool(c[-1] < PDL),
                    ret_close=(c[-1]/lvl - 1)*1e4))
rt = pd.DataFrame(rows)
e1, e2 = rt[rt.ep==1], rt[rt.ep>=2]
print("=== A) MULTIPLE TOUCHES PER DAY (all touch episodes) ===")
print(f"episodes {len(rt)} | 1st {len(e1)} | 2nd+ {len(e2)} | days w/ >=1 touch {rt.date.nunique()} | days w/ re-touch {rt[rt.ep==2].date.nunique()}")
print("\nepisode-count per day (max over sides):")
print(rt.groupby('date').ep.max().value_counts().sort_index())
print("\n--- depth: 1st vs 2nd+ episodes (YR units) ---")
for nm, sub in (("1st", e1), ("2nd+", e2)):
    print(f"{nm}: n={len(sub)}  d30 med={sub.d30.median():.3f}  dcl med={sub.dcl.median():.3f}  dcl mean={sub.dcl.mean():.3f}  P(run day)={sub.run.mean():.4f}")
print("\n2nd+ episode fade proxy, median ret touch->close (bps):")
print(e2.groupby('side').ret_close.median().round(1))

def bucket(m):
    if m < 30:   return "09:31-10:00"
    if m < 90:   return "10:00-11:00"
    if m < 270:  return "11:00-14:00"
    if m < 330:  return "14:00-15:00"
    return "15:00-16:00"
pdh2 = base[base.side=="PDH"].copy(); pdh2["wb"] = pdh2.touch_min.map(bucket)
print("\n=== B) RVOL DISCRIMINATION BY TOUCH TIME (PDH) ===")
g = pdh2.groupby("wb").agg(n=("rvol_touch","size"), med_rvol=("rvol_touch","median"),
      rvol_corr=("rvol_touch", lambda s: s.corr(pdh2.loc[s.index,"dHL_close"], method="spearman")),
      cumrvol_corr=("cumrvol30", lambda s: s.corr(pdh2.loc[s.index,"dHL_close"], method="spearman")))
print(g.round(3))

gaps = ev[ev.gap_over_open]
print("\n=== C) GAP-OPEN RECONCILIATION ===")
print("gap events:", len(gaps), "| unique gap dates:", gaps.date.nunique())
print("by side:", dict(gaps.groupby('side').size()))
print("events per gap day:", dict(gaps.groupby('date').size().value_counts()))

In [ ]:
# --- Cell 11: Stage 2/3 audit --- missing features (VWAP dist, runway, lags, touch count) + ADF ---
new_rows = []
for y in sorted(BYEAR):
    s, e = datetime(y,1,1), datetime(y,12,31)
    mh = qb.history(spy, s, e, Resolution.MINUTE)
    m = mh.loc[spy].copy(); m.index = to_ny(m.index); m = m.sort_index()
    m = m.between_time("09:31","16:00")
    m["d"] = [ts.date() for ts in m.index]
    for d, day in m.groupby("d"):
        if d not in dbars.index: continue
        row = dbars.loc[d]
        PDH, PDL, YR = row["high"], row["low"], row["YR"]
        if not (np.isfinite(YR) and YR > 0): continue
        h = day["high"].values; l = day["low"].values; c = day["close"].values
        v = day["volume"].values; t = day.index; n = len(t)
        tp = (h + l + c) / 3.0
        vwap = np.cumsum(tp * v) / np.maximum(np.cumsum(v), 1.0)   # session VWAP, causal
        for side, lvl in (("PDH", PDH), ("PDL", PDL)):
            above = side == "PDH"
            mask = (h >= lvl) if above else (l <= lvl)
            if not mask.any(): continue
            i0 = int(np.argmax(mask))
            idx = np.flatnonzero(mask)
            n_ep = len(np.split(idx, np.where(np.diff(idx) > 1)[0] + 1))
            new_rows.append(dict(date=d, side=side, min_left=(t[-1]-t[i0]).total_seconds()/60.0,
                vwap_dist_yr=(c[i0] - vwap[i0]) / YR, touch_count=n_ep))
nf = pd.DataFrame(new_rows)

ev2 = ev.sort_values(["side", "date"]).copy()
ev2["lag_dHL"] = ev2.groupby("side").dHL_close.shift(1)      # yesterday's same-side invasion depth
base2 = base.merge(nf, on=["date", "side"], how="left").merge(
    ev2[["date", "side", "lag_dHL"]], on=["date", "side"], how="left")
pdh3 = base2[base2.side == "PDH"]

print("=== STAGE 2 additions: correlation with dHL_close (Spearman, PDH) ===")
for col in ["min_left", "vwap_dist_yr", "lag_dHL", "touch_count", "YR"]:
    sub = pdh3[[col, "dHL_close"]].dropna()
    r = sub[col].corr(sub["dHL_close"], method="spearman") if len(sub) > 10 else np.nan
    print(f"  {col:14s} r = {r:+.3f}   (n={len(sub)})")
print("\nrunway check: corr(min_left, dHL) should mirror touch_min (-0.889) ->", 
      round(pdh3.min_left.corr(pdh3.dHL_close, method='spearman'), 3))
print("\nlag persistence: median dHL when yesterday was deep (>=0.87) vs shallow (<0.87):")
print(pdh3.groupby(pdh3.lag_dHL >= 0.87).dHL_close.median().round(3))
print("\ntouch_count distribution (first-touch days, PDH):", dict(pdh3.touch_count.value_counts().sort_index()))

print("\n=== STAGE 3: stationarity / stability ===")
try:
    from statsmodels.tsa.stattools import adfuller
    for nm, ser in (("dHL_close (PDH)", pdh3.dHL_close.dropna()),
                    ("dvix_to_touch (PDH)", pdh3.dvix_to_touch.dropna()),
                    ("touch_min (PDH)", pdh3.touch_min.dropna())):
        stat, pval, *_ = adfuller(ser.values, autolag="AIC")
        print(f"  ADF {nm:22s} stat={stat:7.3f}  p={pval:.2e}  -> stationary: {pval < 0.05}")
except ImportError:
    print("  statsmodels unavailable - using year-stability table only")
print("\nrelationship stability by year (Spearman vs dHL_close, PDH):")
stab = pdh3.assign(yr=pdh3.date.map(lambda d: d.year)).groupby("yr").apply(
    lambda g: pd.Series({"n": len(g),
        "time": g.touch_min.corr(g.dHL_close, method="spearman"),
        "dvix": g.dvix_to_touch.corr(g.dHL_close, method="spearman"),
        "vwap": g.vwap_dist_yr.corr(g.dHL_close, method="spearman")}), include_groups=False)
print(stab.round(3))
print("\n=== ZERO-LOOKAHEAD AUDIT (schema triage) ===")
print("CAUSAL at touch (gate-eligible): touch_min, min_left, YR, range14, vix_prev, dvix_to_touch,")
print("  vix_touch, rvol_touch, gap_over_open, lag_dHL, vwap_dist_yr, touch_count")
print("FORWARD-LOOKING (diagnostics only, NOT gates): cumrvol30, prepost5, imb, dvix_to_ext,")
print("  dHL_* horizons, tte_min, time_above_min, acc20, recross, mae_yr, rej30, dclass, ret_*")
print("NOTE: the 'cumulative retrace >= x*YR in 30-60 min' entry is causal WITH DELAYED ENTRY -")
print("  you observe the retrace, then enter; fill is x*YR below the touch, not at the touch.")

In [ ]:
# --- Cell 12: incremental information (semi-partial rank correlations, PDH) ---
cols = ["dHL_close", "touch_min", "vwap_dist_yr", "dvix_to_touch", "rvol_touch"]
rk = pdh3[cols].dropna().rank()
print("n (pairwise-complete):", len(rk))

def semipartial(feat, controls):
    y = rk["dHL_close"].values
    Xc = np.column_stack([rk[c].values for c in controls] + [np.ones(len(rk))])
    resid = y - Xc @ np.linalg.lstsq(Xc, y, rcond=None)[0]
    xf = rk[feat].values
    Xf = np.column_stack([rk[c].values for c in controls] + [np.ones(len(rk))])
    rf = xf - Xf @ np.linalg.lstsq(Xf, xf, rcond=None)[0]
    return np.corrcoef(resid, rf)[0, 1]

print("\nraw rank corr vs dHL:")
for f in cols[1:]:
    print(f"  {f:14s} {rk[f].corr(rk['dHL_close']):+.3f}")
print("\nsemi-partial (feature's OWN information after removing touch_min = runway):")
for f in ["vwap_dist_yr", "dvix_to_touch", "rvol_touch"]:
    print(f"  {f:14s} {semipartial(f, ['touch_min']):+.3f}")

def r2(controls):
    y = rk["dHL_close"].values
    X = np.column_stack([rk[c].values for c in controls] + [np.ones(len(rk))])
    pred = X @ np.linalg.lstsq(X, y, rcond=None)[0]
    ss = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return ss

print("\ntogether: explained variance in dHL (rank R^2)")
print(f"  touch_min only              : {r2(['touch_min']):.3f}")
print(f"  + vwap_dist                 : {r2(['touch_min', 'vwap_dist_yr']):.3f}")
print(f"  + vwap_dist + dvix_to_touch : {r2(['touch_min', 'vwap_dist_yr', 'dvix_to_touch']):.3f}")
print(f"  dvix + vwap WITHOUT runway  : {r2(['dvix_to_touch', 'vwap_dist_yr']):.3f}")

In [ ]:
# --- Cell 13: Phase 0b trade simulator + DISCOVERY grid (SPY 2021-2023 only) ---
THETAS = [0.10, 0.15, 0.20, 0.30]
TRIG_WIN = 60          # trigger arms within 60 min of touch
DISC_YEARS = (2021, 2022, 2023)

def run_sim(sym, dbar, years, thetas=THETAS, vixser=None):
    recs = []
    for y in years:
        s, e = datetime(y,1,1), datetime(y,12,31)
        mh = qb.history(sym, s, e, Resolution.MINUTE)
        if not len(mh): continue
        m = mh.loc[sym].copy(); m.index = to_ny(m.index); m = m.sort_index()
        m = m.between_time("09:31","16:00")
        m["d"] = [ts.date() for ts in m.index]
        for d, day in m.groupby("d"):
            if d not in dbar.index: continue
            row = dbar.loc[d]
            PDH, PDL, YR = row["high"], row["low"], row["YR"]
            if not (np.isfinite(YR) and YR > 0): continue
            h = day["high"].values; l = day["low"].values; c = day["close"].values
            o = day["open"].values; t = day.index; n = len(t)
            cut = t[0].replace(hour=15, minute=45)
            iexit = min(int(np.searchsorted(t, cut)), n-1)
            vixp = row["vix_prev_day"] if "vix_prev_day" in dbar.columns else np.nan
            for side, lvl in (("PDH",PDH), ("PDL",PDL)):
                above = side == "PDH"
                if (o[0] >= PDH) if above else (o[0] <= PDL): continue   # gap-open days excluded
                mask = (h >= lvl) if above else (l <= lvl)
                if not mask.any(): continue
                i0 = int(np.argmax(mask))
                vt = vixser.asof(t[i0]) if vixser is not None else np.nan
                dvix = (vt - vixp) if (np.isfinite(vt) and np.isfinite(vixp)) else np.nan
                path = l if above else h
                ext = np.minimum.accumulate(path[i0:]) if above else np.maximum.accumulate(path[i0:])
                depth = ((lvl - ext)/YR) if above else ((ext - lvl)/YR)
                w = min(len(depth), TRIG_WIN + 1)
                for th in thetas:
                    k = np.flatnonzero(depth[:w] >= th)
                    if len(k) == 0: continue
                    rel = int(k[0]); j = i0 + rel
                    if j > iexit: continue
                    entry = lvl - depth[rel]*YR if above else lvl + depth[rel]*YR  # conservative fill
                    hi_run = np.maximum.accumulate(h[j:iexit+1])                   # paths for stop timing
                    lo_run = np.minimum.accumulate(l[j:iexit+1])
                    px = c[iexit]
                    gross = ((entry - px)/entry*1e4) if above else ((px - entry)/entry*1e4)
                    mae = ((hi_run.max()-entry)/YR) if above else ((entry-lo_run.min())/YR)
                    recs.append(dict(date=d, side=side, theta=th, gross=gross, mae_after=mae,
                        dvix=dvix, min_left=(t[-1]-t[j]).total_seconds()/60.0,
                        touch_min=(t[i0]-t[0]).total_seconds()/60.0, dur=iexit-j, yr=YR,
                        hi_run=hi_run.copy(), lo_run=lo_run.copy(), entry=entry, above=above))
    return pd.DataFrame(recs)

disc = run_sim(spy, dbars, DISC_YEARS, vixser=vix_min)
print("discovery trades (trigger-only, no stop yet):", len(disc))

def cfg_table(df):
    rows = []
    for th in THETAS:
        for gname, g in (("trigger-only", df.theta==th),
                         ("+dvix>0", (df.theta==th)&(df.dvix>0)),
                         ("+dvix+runway", (df.theta==th)&(df.dvix>0)&(df.min_left>=120))):
            sub = df[g]
            if len(sub)==0: continue
            rows.append(dict(config=f"th={th:.2f} {gname}", n=len(sub),
                med_gross=sub.gross.median(), mean_gross=sub.gross.mean(),
                q25=sub.gross.quantile(.25), mae_q75=sub.mae_after.quantile(.75)))
    return pd.DataFrame(rows).set_index("config")

tab = cfg_table(disc)
print(tab.round(1))
ok = tab[tab.n >= 100]
pick = ok.med_gross.idxmax()
print("\nPRE-REGISTERED PICK (n>=100, max median gross):", pick)
print("\nstatic sanity: median dHL_close | dHL_60 >= th (unconditional median = 0.87):")
for th in THETAS:
    cond = pdh.loc[pdh.dHL_60 >= th, "dHL_close"]
    print(f"  th={th:.2f}: median {cond.median():.3f}  (n={len(cond)})")

In [ ]:
# --- Cell 14: OOS VALIDATION (2024-2026) with discovery-calibrated stop ---
VAL_YEARS = (2024, 2025, 2026)
pick_theta = 0.10
gmask_d = (disc.theta==pick_theta) & (disc.dvix>0) & (disc.min_left>=120)
s_star = float(disc[gmask_d].mae_after.quantile(.75))
print(f"config: theta={pick_theta}, gates dvix>0 & runway>=120min | stop s* = {s_star:.3f} x YR (discovery MAE q75)")

val = run_sim(spy, dbars, VAL_YEARS, thetas=[pick_theta], vixser=vix_min)
val = val[(val.dvix>0) & (val.min_left>=120)].copy()
print("validation trades (pre-stop):", len(val))

def apply_stop(df, s):
    g, stopped = [], []
    for _, r in df.iterrows():
        s_px = r.entry + s*r.yr if r.above else r.entry - s*r.yr
        run = r.hi_run if r.above else r.lo_run
        hit = np.flatnonzero(run >= s_px) if r.above else np.flatnonzero(run <= s_px)
        if len(hit):
            g.append(-s*r.yr/r.entry*1e4); stopped.append(True)   # stopped: -s x YR from entry
        else:
            g.append(r.gross); stopped.append(False)
    df = df.copy(); df["gross_stop"] = g; df["stopped"] = stopped
    df["net"] = df.gross_stop - 2.0                                # 2 bps round-trip cost
    return df

disc_g = apply_stop(disc[gmask_d].copy(), s_star)
val_g  = apply_stop(val, s_star)

def report(nm, df):
    t = df.net.mean() / (df.net.std()/np.sqrt(len(df))) if len(df) > 2 else np.nan
    print(f"{nm}: n={len(df)}  stopped={df.stopped.mean():.1%}  med_gross={df.gross_stop.median():+.1f}bps  "
          f"med_net={df.net.median():+.1f}  mean_net={df.net.mean():+.1f}  t={t:+.2f}")
print("\n=== WITH STOP (s* from discovery only) ===")
report("discovery 2021-23", disc_g)
report("VALIDATION 24-26", val_g)
print("\nvalidation by year (net):")
print(val_g.assign(y=val_g.date.map(lambda d: d.year)).groupby("y").agg(
    n=("net","size"), med=("net","median"), mean=("net","mean")).round(1))
print("\nvalidation by side (net):")
print(val_g.groupby("side").agg(n=("net","size"), med=("net","median")).round(1))

print("\n=== PRE-REGISTERED GATES ===")
tv = val_g.net.mean()/(val_g.net.std()/np.sqrt(len(val_g)))
print(f"G1 t>=2 on net (validation):        t={tv:+.2f}  -> {'PASS' if tv>=2 else 'FAIL'}")
print(f"G2 median gross >= 8bps (valid.):   {val_g.gross_stop.median():+.1f}  -> {'PASS' if val_g.gross_stop.median()>=8 else 'FAIL'}")
allg = pd.concat([disc_g, val_g]); allg["y"] = allg.date.map(lambda d: d.year)
yrpos = (allg.groupby("y").net.median() > 0).sum()
print(f"G3 net median >0 in >=4/6 years:    {yrpos}/6  -> {'PASS' if yrpos>=4 else 'FAIL'}")
print(f"G4 stop-adj mean net > 0 (valid.):  {val_g.net.mean():+.1f}  -> {'PASS' if val_g.net.mean()>0 else 'FAIL'}")

print("\nbenchmark: unconditional non-gap touch->close median (bps): PDH", round(pdh.ret_close_bps.median(),1),
      "| PDL", round(pdl.ret_close_bps.median(),1),
      "| entry offset cost ~0.1 x YR =", round((0.1*val_g.yr/val_g.entry*1e4).median(),1), "bps")

In [ ]:
# --- Cell 15: QQQ replication + fair naive benchmark (same periods) ---
qqq = qb.add_equity("QQQ", Resolution.MINUTE, data_normalization_mode=DataNormalizationMode.RAW).symbol
qd_ = qb.history(qqq, START, END, Resolution.DAILY)
qdb = qd_.loc[qqq].copy(); qdb.index = to_ny(qdb.index)
qdb["date"] = [d.date() for d in qdb.index]; qdb = qdb.set_index("date").sort_index()
qdb["YR"] = qdb["high"] - qdb["low"]
qdb["range14"] = qdb["YR"].rolling(14).mean()
qdb["vix_prev_day"] = np.nan

qdsc = run_sim(qqq, qdb, DISC_YEARS, thetas=[0.10])
qdsc = qdsc[qdsc.min_left >= 120].copy()          # dvix unavailable on QQQ -> runway gate only
s_q = float(qdsc.mae_after.quantile(.75))
qv = run_sim(qqq, qdb, VAL_YEARS, thetas=[0.10])
qv = qv[qv.min_left >= 120].copy()
qv_g = apply_stop(qv, s_q)
tvq = qv_g.net.mean()/(qv_g.net.std()/np.sqrt(len(qv_g))) if len(qv_g) else np.nan
print(f"QQQ replication: disc n={len(qdsc)} stop={s_q:.3f}xYR | VAL n={len(qv_g)} stopped={qv_g.stopped.mean():.1%} "
      f"med_net={qv_g.net.median():+.1f} mean_net={qv_g.net.mean():+.1f} t={tvq:+.1f}")
print(qv_g.groupby("side").agg(n=("net","size"), med=("net","median")).round(1))

print("\n=== FAIR BENCHMARK: naive fade-from-touch, same years (2024-26), no gates, no stop, hold to close ===")
for nm, sub in (("PDH(short)", pdh), ("PDL(long)", pdl)):
    v = sub[sub.date.map(lambda d: d.year).isin(VAL_YEARS)]
    naive = (-v.ret_close_bps) if nm.startswith("PDH") else v.ret_close_bps
    print(f"  naive {nm}: n={len(v)}  med={naive.median():+.1f}  mean={naive.mean():+.1f}  p05={naive.quantile(.05):+.1f}")
print("  engineered fade VAL med net: PDH +34.4 | PDL +16.6  (pays ~8bps entry offset + stop premium)")
print("\n=== tail check: worst single trade (net), validation ===")
print("SPY:", round(val_g.net.min(),1), "bps |", val_g.loc[val_g.net.idxmin(), "date"], val_g.loc[val_g.net.idxmin(), "side"])
print("QQQ:", round(qv_g.net.min(),1), "bps |", qv_g.loc[qv_g.net.idxmin(), "date"], qv_g.loc[qv_g.net.idxmin(), "side"])

In [ ]:
# --- Cell 16: DURATION-EXIT STUDY (picked config, close paths per trade) ---
def run_sim2(sym, dbar, years, theta, vixser=None):
    recs = []
    for y in years:
        s, e = datetime(y,1,1), datetime(y,12,31)
        mh = qb.history(sym, s, e, Resolution.MINUTE)
        if not len(mh): continue
        m = mh.loc[sym].copy(); m.index = to_ny(m.index); m = m.sort_index()
        m = m.between_time("09:31","16:00")
        m["d"] = [ts.date() for ts in m.index]
        for d, day in m.groupby("d"):
            if d not in dbar.index: continue
            row = dbar.loc[d]
            PDH, PDL, YR = row["high"], row["low"], row["YR"]
            if not (np.isfinite(YR) and YR > 0): continue
            h = day["high"].values; l = day["low"].values; c = day["close"].values
            o = day["open"].values; t = day.index; n = len(t)
            cut = t[0].replace(hour=15, minute=45)
            iexit = min(int(np.searchsorted(t, cut)), n-1)
            vixp = row["vix_prev_day"] if "vix_prev_day" in dbar.columns else np.nan
            for side, lvl in (("PDH",PDH), ("PDL",PDL)):
                above = side == "PDH"
                if (o[0] >= PDH) if above else (o[0] <= PDL): continue
                mask = (h >= lvl) if above else (l <= lvl)
                if not mask.any(): continue
                i0 = int(np.argmax(mask))
                vt = vixser.asof(t[i0]) if vixser is not None else np.nan
                dvix = (vt - vixp) if (np.isfinite(vt) and np.isfinite(vixp)) else np.nan
                path = l if above else h
                ext = np.minimum.accumulate(path[i0:]) if above else np.maximum.accumulate(path[i0:])
                depth = ((lvl - ext)/YR) if above else ((ext - lvl)/YR)
                w = min(len(depth), 61)
                k = np.flatnonzero(depth[:w] >= theta)
                if len(k) == 0: continue
                rel = int(k[0]); j = i0 + rel
                if j > iexit: continue
                entry = lvl - depth[rel]*YR if above else lvl + depth[rel]*YR
                recs.append(dict(date=d, side=side, yr=YR, entry=entry, above=above, dvix=dvix,
                    min_left=(t[-1]-t[j]).total_seconds()/60.0, rel=rel,
                    hi_run=np.maximum.accumulate(h[j:iexit+1]).copy(),
                    lo_run=np.minimum.accumulate(l[j:iexit+1]).copy(),
                    c_path=c[j:iexit+1].copy()))
    return pd.DataFrame(recs)

def gate(df, spy=True):
    g = df.min_left >= 120
    if spy: g &= (df.dvix > 0)
    return df[g].copy()

D = {"SPY_disc": gate(run_sim2(spy, dbars, DISC_YEARS, 0.10, vix_min)),
     "SPY_val":  gate(run_sim2(spy, dbars, VAL_YEARS, 0.10, vix_min)),
     "QQQ_disc": gate(run_sim2(qqq, qdb, DISC_YEARS, 0.10), spy=False),
     "QQQ_val":  gate(run_sim2(qqq, qdb, VAL_YEARS, 0.10), spy=False)}
for k, v in D.items(): print(k, "trades:", len(v))

S = {"SPY": 0.123, "QQQ": 0.133}
MGRID = [15, 30, 45, 60, 90, 120, 180, 240]

def gross_at_M(r, s, M):
    L = len(r.c_path); idx = min(M, L-1)
    stop_px = r.entry + s*r.yr if r.above else r.entry - s*r.yr
    run = r.hi_run if r.above else r.lo_run
    seg = run[:idx+1]
    hit = np.flatnonzero(seg >= stop_px) if r.above else np.flatnonzero(seg <= stop_px)
    px = stop_px[0]*0 + stop_px if len(hit) else r.c_path[idx]
    g = ((r.entry - px)/r.entry*1e4) if r.above else ((px - r.entry)/r.entry*1e4)
    return g, idx + 1

def dur_table(df, s):
    rows = []
    for M in MGRID + [10**9]:
        out = [gross_at_M(r, s, M) for r in df.itertuples()]
        g = np.array([x[0] for x in out]); ex = np.array([x[1] for x in out])
        rows.append(dict(M=("hold" if M > 10**8 else M), med_net=np.median(g)-2.0,
            mean_net=g.mean()-2.0, med_exp=np.median(ex), net_per_hr=((g.mean()-2.0)/ex.mean()*60)))
    return pd.DataFrame(rows).set_index("M")

print("\n=== SPY DISCOVERY: exit at entry+M minutes (stop always active) ===")
t_disc = dur_table(D["SPY_disc"], S["SPY"])
print(t_disc.round(1))
hold_med = t_disc.loc["hold", "med_net"]
cands = t_disc[(t_disc.index != "hold") & (t_disc.med_net >= 0.9*hold_med)]
M_STAR = int(cands.index[0]) if len(cands) else None
print(f"\nPICK (smallest M with med_net >= 90% of hold): M* = {M_STAR if M_STAR else 'hold'}")

In [ ]:
# --- Cell 17: duration-exit accounting (fixed) ---
def gross_at_M(r, s, M):
    L = len(r.c_path); idx = min(M, L-1)
    stop_px = r.entry + s*r.yr if r.above else r.entry - s*r.yr
    run = r.hi_run if r.above else r.lo_run
    seg = run[:idx+1]
    hit = np.flatnonzero(seg >= stop_px) if r.above else np.flatnonzero(seg <= stop_px)
    px = stop_px if len(hit) else r.c_path[idx]
    g = ((r.entry - px)/r.entry*1e4) if r.above else ((px - r.entry)/r.entry*1e4)
    return g, idx + 1

def dur_table(df, s):
    rows = []
    for M in MGRID + [10**9]:
        out = [gross_at_M(r, s, M) for r in df.itertuples()]
        g = np.array([x[0] for x in out]); ex = np.array([x[1] for x in out])
        rows.append(dict(M=("hold" if M > 10**8 else M), med_net=np.median(g)-2.0,
            mean_net=g.mean()-2.0, med_exp=np.median(ex), net_per_hr=((g.mean()-2.0)/ex.mean()*60)))
    return pd.DataFrame(rows).set_index("M")

print("=== SPY DISCOVERY: exit at entry+M (stop always active), net bps ===")
t_disc = dur_table(D["SPY_disc"], S["SPY"])
print(t_disc.round(1))
hold_med = t_disc.loc["hold", "med_net"]
cands = t_disc[(t_disc.index != "hold") & (t_disc.med_net >= 0.9*hold_med)]
M_STAR = int(cands.index[0]) if len(cands) else None
print(f"\nPICK rule (smallest M with med_net >= 90% of hold): M* = {M_STAR}")

print("\n=== VALIDATION of M* (and full curve for context) ===")
for k in ["SPY_val", "QQQ_disc", "QQQ_val"]:
    print(f"\n{k}:")
    print(dur_table(D[k], S[k.split("_")[0]]).round(1))

In [ ]:
# --- Cell 18: closing diagnostics for the exit question (discovery only, diagnostic-only) ---
for k in ["SPY_disc", "QQQ_disc"]:
    df = D[k]; s = S[k.split("_")[0]]
    g60 = np.array([gross_at_M(r, s, 60)[0] for r in df.itertuples()])
    gh  = np.array([gross_at_M(r, s, 10**9)[0] for r in df.itertuples()])
    los = g60 < 0
    print(f"{k}: n_losers@60 = {los.sum()} ({100*los.mean():.0f}%)  "
          f"med gross 60->hold for losers = {np.median(gh[los]) - np.median(g60[los]):+.1f} bps further  "
          f"(med gross(hold) losers = {np.median(gh[los]):+.1f})")
    fast = df.rel <= 15
    print(f"   trigger speed: fast(<=15m) n={fast.sum()} med gross(hold)={np.median(gh[fast.values]):+.1f}  "
          f"slow n={(~fast).sum()} med={np.median(gh[~fast.values]):+.1f}")

In [ ]:
# --- Cell 19: verify LEAN entry prices against daily context (levels stale?) ---
lean = [
    ("2021-01-11", "QQQ", "PDL", 315.58), ("2021-01-12", "QQQ", "PDL", 313.43),
    ("2021-01-12", "SPY", "PDL", 377.97), ("2021-01-14", "QQQ", "PDH", 316.55),
    ("2021-01-15", "QQQ", "PDL", 313.36), ("2021-01-21", "SPY", "PDH", 384.39),
    ("2021-01-25", "SPY", "PDH", 383.71), ("2021-01-25", "QQQ", "PDL", 323.66)]
for dt, tk, side, px in lean:
    d = pd.Timestamp(dt).date()
    db = dbars if tk == "SPY" else qdb
    i = db.index.searchsorted(d)
    for tag, r in (("T-1", db.iloc[i-1]), ("T-2", db.iloc[i-2])):
        YR = r["YR"]
        lvl = r["high"] if side == "PDH" else r["low"]
        trig = lvl - 0.10*YR if side == "PDH" else lvl + 0.10*YR
        ok = abs(trig - px) < 0.15*YR
        print(f"{dt} {tk} {side} fill={px:7.2f} | {tag}({r.name}) H={r['high']:7.2f} L={r['low']:7.2f} "
              f"YR={YR:5.2f} -> trig={trig:7.2f} {'> MATCH' if ok else ''}")
    print()

In [ ]:
# --- Cell 20: notebook trades Jan 11-25 vs LEAN order book ---
nb = []
for k in ["SPY_disc", "QQQ_disc"]:
    tk = k.split("_")[0]
    for r in D[k].itertuples():
        if pd.Timestamp("2021-01-11").date() <= r.date <= pd.Timestamp("2021-01-25").date():
            nb.append((r.date, tk, r.side))
print("NOTEBOOK trades Jan 11-25:", len(nb))
for t in sorted(nb): print("  ", t[0], t[1], t[2])
print("\nLEAN entries Jan 11-25 (from order book): 8 ->")
print("   01-11 QQQ PDL | 01-12 QQQ PDL | 01-12 SPY PDL | 01-14 QQQ PDH | "
      "01-15 QQQ PDL | 01-21 SPY PDH | 01-25 SPY PDH | 01-25 QQQ PDL")

In [ ]:
# --- Cell 21: LOOK-AHEAD AUDIT of the reference simulator ---
# Cell 2 built dbars from RAW daily history: dbars.loc[d] = day d's OWN OHLC (YR = same-day range), no shift.
# Cells 3/4/13/16 then used row=dbar.loc[d] as 'prior-day context' for day-d trades.
# If so, 'PDL' = TODAY'S terminal low -> touch guaranteed by definition, and every FAILED
# bounce (which a real trader takes and loses on) is silently excluded. Test:
# run the IDENTICAL simulator on true prior-day levels (shift(1)) and compare.
import numpy as np, pandas as pd
from datetime import datetime as dt

def build_ctx(sym, lag):
    daily = qb.history(sym, dt(2020,11,1), dt(2026,10,1), Resolution.DAILY)
    b = daily.loc[sym].copy(); b.index = to_ny(b.index)
    b["date"] = [x.date() for x in b.index]
    b = b.set_index("date").sort_index()
    b["YR"] = b["high"] - b["low"]
    out = pd.DataFrame(index=b.index)
    if lag:   # TRUE prior-day context: row d carries day d-1's bar
        out["high"], out["low"], out["YR"] = b["high"].shift(1), b["low"].shift(1), b["YR"].shift(1)
    else:     # the notebook's cell-2 construction: same-day bar
        out["high"], out["low"], out["YR"] = b["high"], b["low"], b["YR"]
    return out

qqq = qb.add_equity("QQQ", Resolution.MINUTE, data_normalization_mode=DataNormalizationMode.RAW).symbol

def sim_both(sym, ctx_raw, ctx_lag, theta=0.10, years=range(2021,2027)):
    out = {"raw": [], "lag": []}
    for y in years:
        mh = qb.history(sym, dt(y,1,1), dt(y,12,31), Resolution.MINUTE)
        if not len(mh): continue
        m = mh.loc[sym].copy(); m.index = to_ny(m.index); m = m.sort_index()
        m = m.between_time("09:31","16:00")
        m["d"] = [ts.date() for ts in m.index]
        for d, day in m.groupby("d"):
            for tag, dbar in (("raw", ctx_raw), ("lag", ctx_lag)):
                if d not in dbar.index: continue
                row = dbar.loc[d]
                PDH, PDL, YR = row["high"], row["low"], row["YR"]
                if not (np.isfinite(YR) and YR > 0): continue
                h = day["high"].values; l = day["low"].values
                c = day["close"].values; o = day["open"].values
                t = day.index; n = len(t)
                cut = t[0].replace(hour=15, minute=45)
                iexit = min(int(np.searchsorted(t, cut)), n-1)
                for side, lvl in (("PDH",PDH), ("PDL",PDL)):
                    above = side == "PDH"
                    if (o[0] >= PDH) if above else (o[0] <= PDL): continue
                    mask = (h >= lvl) if above else (l <= lvl)
                    if not mask.any(): continue
                    i0 = int(np.argmax(mask))
                    path = l if above else h
                    ext = np.minimum.accumulate(path[i0:]) if above else np.maximum.accumulate(path[i0:])
                    depth = ((lvl - ext)/YR) if above else ((ext - lvl)/YR)
                    w = min(len(depth), 61)
                    k = np.flatnonzero(depth[:w] >= theta)
                    if len(k) == 0: continue
                    rel = int(k[0]); j = i0 + rel
                    if j > iexit: continue
                    entry = lvl - depth[rel]*YR if above else lvl + depth[rel]*YR
                    px = c[iexit]
                    gross = ((entry - px)/entry*1e4) if above else ((px - entry)/entry*1e4)
                    out[tag].append(dict(date=d, side=side, gross=gross))
    return pd.DataFrame(out["raw"]), pd.DataFrame(out["lag"])

ctx_raw_spy, ctx_lag_spy = build_ctx(spy, False), build_ctx(spy, True)
ctx_raw_qqq, ctx_lag_qqq = build_ctx(qqq, False), build_ctx(qqq, True)

summary = {}
for nm, sym, cr, cl in (("SPY",spy,ctx_raw_spy,ctx_lag_spy), ("QQQ",qqq,ctx_raw_qqq,ctx_lag_qqq)):
    r_raw, r_lag = sim_both(sym, cr, cl)
    summary[nm] = dict(
        n_raw=len(r_raw), n_lag=len(r_lag),
        med_bps_raw=r_raw.gross.median() if len(r_raw) else np.nan,
        med_bps_lag=r_lag.gross.median() if len(r_lag) else np.nan,
        q25_bps_lag=r_lag.gross.quantile(.25) if len(r_lag) else np.nan,
        mean_bps_lag=r_lag.gross.mean() if len(r_lag) else np.nan)
    for side in ("PDH","PDL"):
        s = r_lag[r_lag.side==side]
        if len(s): summary[nm][f"n_lag_{side}"] = len(s); summary[nm][f"med_lag_{side}"] = s.gross.median()
print(pd.DataFrame(summary).T.round(1))
print("\nLEAN reference for the same period (theta=0.10 + its gates): ~900 entries")

# Forensic: the notebook's 'impossible' trade — 2021-01-13 QQQ PDL.
# Prior-day (Jan-12) low = 311.20 was NEVER touched on Jan-13 (own low 313.57).
tq = D["QQQ_disc"]
key = [x.strftime("%Y-%m-%d") for x in tq["date"]]
r = tq[(pd.Series(key) == "2021-01-13").values & (tq["side"] == "PDL")]
print("\nnotebook 2021-01-13 QQQ PDL rows (entry/yr):")
print(r[["side","entry","yr"]].to_string() if len(r) else "none")
print("QQQ daily low 2021-01-12 (true prior-day PDL):", ctx_lag_qqq.loc[dt(2021,1,13).date(), "low"])
print("QQQ daily low 2021-01-13 (own-day low):        ", ctx_raw_qqq.loc[dt(2021,1,13).date(), "low"])


# POST-MORTEM (cell 21 audit): the reference simulator had a look-ahead bug

**Bug:** Cell 2 built `dbars` from raw daily history — `dbars.loc[d]` is day **d's own** OHLC (`YR = high − low` same-day, no shift). Cells 3/4/13/16 then used `row = dbar.loc[d]` as "prior-day context" for day-d trades. So "PDH/PDL" were **today's terminal high/low** — knowable only at the close. Touching the day's own low is guaranteed by definition, and every *failed* bounce (which a real trader takes and loses on) was silently excluded because `i0` jumps straight to the bar that sets the terminal extreme.

**Evidence (cell 21):**
- Forensic: notebook trade 2021-01-13 QQQ PDL entry = 313.90 = **own-day low 313.57 + 0.10 × own-day YR 3.29**; the true prior-day low (311.20, Jan-12) was never touched that day. Same-day-level identity holds by construction.
- Identical simulator, true prior-day (shift-1) levels, θ=0.10, no gates, 2021–2026: SPY 2,338→**615** trades, QQQ 2,414→**646**; total 1,261 vs LEAN's ~900 gated entries — reconciles once LEAN's gates (dvix, 14:00 cutoff) are applied. The "missing ~70%" of trades never existed.
- Honest expectancy of the fade-first-touch structure at θ=0.10, exit 15:45, **before costs**: SPY median **−7.8 bps** (q25 −43.8), QQQ median **−16.3 bps** (q25 −65.8); negative on both sides of both symbols (PDH −6.7/−17.4, PDL −9.2/−12.4).

**Consequences:**
1. All Phase-0a/0b results (event study, discovery grid, pre-registered pick, duration study, 2,882-trade simulation, per-trade bps distributions) are **void** — they measured the arithmetic of conditioning on the day's terminal extreme, not a tradable signal.
2. The LEAN backtest (~900 trades, levels verified against T-1 in cell 19) is the only honest artifact. Its premise — fading first touches of prior-day extremes — is **refuted in this form** on SPY/QQQ 2021–2026: negative gross expectancy before costs.
3. The cell-19 anomalies and the Jan 11–25 trade-count/side mismatches are fully explained by this bug (plus next-bar-open fill mechanics), not by LEAN misbehavior.

**What would need to be true to continue:** a conditioning variable measurable in real time at the touch (rejection quality, VIX regime, overnight gap, volume impulse) would have to flip the −8/−16 bps base expectancy positive AND survive costs. That is a new discovery process — run only on true prior-day levels, ideally with the multiple-touch structure a real trader faces (every new extreme re-arms), not the terminal-extreme single-touch used here.


In [ ]:
# --- Cell 23: THE 15-MINUTE TEST (implementation in scalp15_test.py; rules pre-registered there) ---
import scalp15_test
scalp = scalp15_test.run()


# Cell 24 — 15-MINUTE TEST VERDICT: **KILLED** (per pre-registered rule in `scalp15_test.py`)

Setup: corrected structure (true prior-day levels, first touch, retrace ≥ 0.10×YR within 60 bars), entry = trigger-bar **close** (real-time), exit = +15 min. 1,287 triggers, 800 sessions, SPY+QQQ, 2021–2026.

| M=15, fade dir (gross bps) | median | mean | win | t | net of 2 bps RT |
|---|---|---|---|---|---|
| QQQ (n=656) | −0.4 | +0.4 | 48.8% | +0.3 | −2.4 / −1.6 |
| SPY (n=631) | −1.6 | −2.2 | 44.7% | **−3.1** | −3.6 / −4.2 |

- **KILL rule hit:** fade median ≤ 0 gross on both symbols. The 15-min window is *flat-to-against* the fade, not a positive-expectancy burst. SPY fade is significantly negative (t = −3.1).
- **Momentum mirror also dead:** SPY +2.2 mean gross overall (t +3.1) but only +1.1 in 2024–26 validation (t +1.1) and +0.2 net of costs; QQQ mirror ≈ 0 everywhere (validation mean −1.2). Fails the FLIP bar (≥+4 gross both symbols, t≥3 validation).
- **Horizon profile (fade median, 5→120 min):** SPY −0.4→−3.2, QQQ −0.8→−2.8. No window works in either direction.
- **Why the 'burst' is untradable:** with the extreme-fill convention (entry at the retrace low), fade median at M=15 is −8.6 (QQQ) / −5.3 (SPY) — i.e., price bounces ~5–9 bps off the retrace extreme within 15 min, but by the time the trigger is *observable* (bar close), the bounce is mostly spent. The reversion happens **before** any real-time entry; what remains after entry is drift ≈ 0 minus costs.
- Conditioners don't rescue it: dvix<0 best cell (QQQ +0.9 mean) < costs; QQQ 14–15h bucket (+10.5, n=38) is a multiple-comparison artifact (SPY same bucket −0.3).
- Even at face value the economics were thin: ~110 triggers/yr/symbol × +2 bps ≈ +20 bps/yr on notional.

**Status:** revival-bar candidate #1 (from card #42792) tested and refuted. The line stays shelved: fade ≤ 0 at every horizon 5–120 min; mirror fails validation and costs. Next candidate needs a conditioner measured *at* the touch with ≥ ~5 bps median gross edge — none found here.

In [ ]:
# --- Cell 25: RE-ARMING (second-and-later-touch) variant, M=15 — closes the last refutation gap ---
import importlib, scalp15_test
importlib.reload(scalp15_test)
rearm = scalp15_test.run_rearm()


# CELL 26 — MASTER RECORD: PDH/PDL RESEARCH LINE (SPY/QQQ, 2021-01 → 2026-09) — **STATUS: CLOSED**

Full narrative: `RESEARCH_RECORD.md` (project file). Artifacts: cells 0–25, `scalp15_test.py`, LEAN runs `796c380f` (no stops) / `d6193ee8` (final, stops), cards #42792 / #42796.

**Verdict — the strategy family is refuted at every horizon, every variant, every conditioner (all gross, pre-registered):**

| variant | result |
|---|---|
| Full-day fade, corrected sim (exit 15:45) | SPY med **−7.8** / QQQ **−16.3** bps |
| LEAN full-day fade with stops (final) | **−13.9% net**, Sharpe −1.0, 15% win, maxDD 17% |
| 15-min scalp, M=15, real-time entry | QQQ −0.4 / SPY **−1.6** (t −3.1); net of 2 bps negative |
| Horizon grid M = 5→120 min | fade −0.4…−3.2 bps everywhere — no window works |
| Momentum mirror (flip) | SPY +2.2 gross → +1.1 validation → **+0.2 net**; QQQ ≈ 0 |
| Conditioners (dvix sign, time-of-day) | best honest cell +0.9 mean < costs; juicy cells are n≈38 noise |
| Re-arming / second-touch (6,232 triggers) | fade −0.5 / −1.1 mean (SPY t −4.1); mirror +0.5 / +1.1 < 2 bps costs |

**The two lessons that generalize beyond this line:**
1. **Look-ahead via context construction.** Using same-day OHLC as "prior-day" levels conditions on terminal extremes — manufactured +37/+51 bps of fake edge and voided every Phase-0a/0b result. Any level/feature must be shift-1 and audited with an own-day-extreme identity check (cell 21 forensic: 2021-01-13 QQQ).
2. **Fill-timing illusion.** The 5–9 bps bounce off the retrace extreme is real but completes *before* the trigger is observable; from a decision-time (trigger-close) entry the residual drift ≈ 0. Reversion studies must enter at decision-time prices, never at extremes-in-hindsight.

**Strategy dead; phenomenon open as feature material — with an honesty caveat.** Corrected facts: true prior-day extremes produce 1,287 first-touch triggers (θ=0.10, 60-bar window) and 6,232 re-arming triggers over 5.75y; post-entry drift at true levels is ~0 to negative in every direction and horizon tested. The rich conditioning statistics from the early event study (invasion depth distribution, VIX/volume/time-of-day effects) were own-day artifacts — **void** — so the phenomenon's *predictability* at true levels is an unproven open question, not a settled fact. Any future use: shift-1 levels only, real-time entry, discovery/validation split, ≥ ~5 bps median gross bar, before any LEAN build. Revival terms live on card #42796.